# North African Stock Markets — Factor Structure Analysis
### MASI · EGX30 · Tunindex | 2015–2024

**Authors:** Younes El Aaroua & Adham El Hayani  
**Institution:** FEG Settat — M1 Actuarial Science, Insurance & Finance  
**Supervisor:** Prof. Abdallah Abarda

---

### Before running
1. Place `MASI.csv`, `EGX30.csv`, `Tunindex.csv` in this folder (download from Investing.com)
2. Run **Cell 1** once to install packages
3. Then run all cells top to bottom (`Kernel → Restart & Run All`)

### What this notebook produces
| Output | Description |
|--------|-------------|
| `log_returns_with_periods.csv` | Daily log-returns labelled by market regime |
| `descriptive_stats.csv` | Summary statistics (Table 1 of paper) |
| `pca_eigenvalues.csv` | PCA variance explained (Table 3 of paper) |
| `acm_coordinates.csv` | ACM category coordinates (Table 5 of paper) |
| `data_overview.png` | EDA dashboard (Appendix A, Figure 1) |
| `pca_scree.png` + others | All PCA figures (Figures 2–4 of paper) |
| `afc_residuals.png` + others | All AFC/ACM figures (Figures 5–6 of paper) |

---
## Step 0 — Install packages *(run once)*

In [ ]:
import subprocess
for pkg in ["pandas","numpy","scipy","statsmodels","matplotlib","scikit-learn","prince"]:
    subprocess.run(["pip","install",pkg,"-q"], check=True)
print("✓ All packages ready")

---
## Step 1 — Imports & configuration

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
import prince, warnings
warnings.filterwarnings("ignore")

# ── Colour scheme ──────────────────────────────────────────
COLORS        = {"MASI":"#533AB7","EGX30":"#D85A30","Tunindex":"#1D9E75"}
PC_COLORS     = ["#E63946","#2A9D8F","#F4A261"]
INDEX_MARKERS = {"MASI":"o","EGX30":"s","Tunindex":"D"}
PERIOD_SPANS  = [
    ("2015-01-01","2019-12-31","#EEEDFE","Pre-COVID (2015–2019)"),
    ("2020-01-01","2021-12-31","#FAECE7","COVID (2020–2021)"),
    ("2022-01-01","2024-12-31","#E1F5EE","Post-COVID (2022–2024)"),
]
PERIODS = {
    "Full Sample\n(2015–2024)": ("2015-01-01","2024-12-31"),
    "Pre-COVID\n(2015–2019)":   ("2015-01-01","2019-12-31"),
    "COVID\n(2020–2021)":       ("2020-01-01","2021-12-31"),
    "Post-COVID\n(2022–2024)":  ("2022-01-01","2024-12-31"),
}
print("✓ Configuration ready")

---
## Step 2 — Load price data from CSV

> Download from [Investing.com](https://www.investing.com) — Daily frequency, 2015–2024

In [ ]:
def load_csv(filepath, name):
    df = pd.read_csv(filepath)
    df["Date"]  = pd.to_datetime(df["Date"])
    df["Price"] = df["Price"].str.replace(",","").astype(float)
    return df.sort_values("Date").set_index("Date")["Price"].rename(name)

prices = pd.DataFrame({
    "MASI"    : load_csv("MASI.csv",     "MASI"),
    "EGX30"   : load_csv("EGX30.csv",    "EGX30"),
    "Tunindex": load_csv("Tunindex.csv", "Tunindex"),
}).loc["2015-01-01":"2024-12-31"].dropna()

prices.index.name = "Date"
prices.to_csv("prices_raw.csv")
print(f"✓ {len(prices)} trading days | {prices.index[0].date()} → {prices.index[-1].date()}")
print(prices.tail(3))

---
## Step 3 — Log-returns, descriptive stats & stationarity

In [ ]:
# ── Log-returns ───────────────────────────────────────────
def label(date):
    if date < pd.Timestamp("2020-01-01"): return "Pre-COVID (2015–2019)"
    elif date <= pd.Timestamp("2021-12-31"): return "COVID (2020–2021)"
    else: return "Post-COVID (2022–2024)"

returns          = np.log(prices / prices.shift(1)).dropna()
returns["Period"]= returns.index.map(label)
returns.to_csv("log_returns_with_periods.csv")
ret = returns[["MASI","EGX30","Tunindex"]]

# ── Descriptive statistics ─────────────────────────────────
rows = []
for col in ret.columns:
    s = ret[col].dropna()
    jb_p = stats.jarque_bera(s).pvalue
    rows.append({"Index":col,"N":len(s),
        "Mean ann.%":round(s.mean()*252*100,3),
        "Std ann.%" :round(s.std()*np.sqrt(252)*100,3),
        "Min%":round(s.min()*100,3),"Max%":round(s.max()*100,3),
        "Skewness":round(stats.skew(s),4),
        "Exc.Kurt":round(stats.kurtosis(s),4),
        "JB p":("<0.001" if jb_p<0.001 else round(jb_p,4)),
        "Normal?":("No" if jb_p<0.05 else "Yes")})
desc_df = pd.DataFrame(rows).set_index("Index")
desc_df.to_csv("descriptive_stats.csv")
print("── Descriptive Statistics ──"); print(desc_df.T.to_string())

# ── ADF stationarity tests ─────────────────────────────────
print("\n── ADF Tests ──")
for col in ret.columns:
    r = adfuller(ret[col].dropna(), autolag="AIC")
    print(f"  {col:12s} ADF={r[0]:8.4f}  p={r[1]:.6f}  → {'Stationary ✓' if r[1]<0.05 else 'NON-stationary ✗'}")

# ── Correlations ───────────────────────────────────────────
corr = ret.corr().round(4)
corr.to_csv("correlation_matrix.csv")
print("\n── Full-sample correlations ──"); print(corr.to_string())

# ── Sub-period stats ───────────────────────────────────────
print("\n── Sub-period breakdown ──")
sp_rows=[]
for lbl,(s,e) in [("Pre-COVID","2015-01-01 2019-12-31"),
                   ("COVID","2020-01-01 2021-12-31"),
                   ("Post-COVID","2022-01-01 2024-12-31")]:
    s,e = lbl, e  # skip — use PERIODS dict
for plabel,(_,_,_,lname) in [(p,PERIOD_SPANS[i]) for i,p in enumerate(["Pre-COVID (2015–2019)","COVID (2020–2021)","Post-COVID (2022–2024)"])]:
    sub = ret[returns["Period"]==plabel]
    print(f"  {plabel} (n={len(sub)}): vol MASI={sub['MASI'].std()*np.sqrt(252)*100:.1f}% "
          f"ρ(M,E)={sub['MASI'].corr(sub['EGX30']):.3f} "
          f"ρ(M,T)={sub['MASI'].corr(sub['Tunindex']):.3f}")

---
## Step 4 — EDA Dashboard Chart

In [ ]:
fig = plt.figure(figsize=(15,20), facecolor="white")
gs  = GridSpec(4,3, figure=fig, hspace=0.52, wspace=0.38,
               left=0.07, right=0.97, top=0.95, bottom=0.05)

# Panel 1: Normalised prices
ax1 = fig.add_subplot(gs[0,:])
norm = (prices/prices.iloc[0])*100
for col,color in COLORS.items():
    ax1.plot(norm.index, norm[col], label=col, color=color, linewidth=1.3)
for s,e,c,lbl in PERIOD_SPANS:
    ax1.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.5, label=lbl)
ax1.set_title("Normalised Price Indices (Base=100, 2015–2024)", fontsize=12, fontweight="bold")
ax1.legend(fontsize=8, ncol=6); ax1.spines[["top","right"]].set_visible(False)

# Panel 2: Daily log-returns
ax2 = fig.add_subplot(gs[1,:])
for col,color in COLORS.items():
    ax2.plot(ret.index, ret[col]*100, color=color, linewidth=0.5, alpha=0.85, label=col)
for s,e,c,_ in PERIOD_SPANS:
    ax2.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.4)
ax2.set_title("Daily Log-Returns (%)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8); ax2.spines[["top","right"]].set_visible(False)

# Panel 3: Return distributions
for i,(col,color) in enumerate(COLORS.items()):
    ax = fig.add_subplot(gs[2,i])
    data = ret[col].dropna()*100
    ax.hist(data, bins=80, color=color, alpha=0.7, density=True, edgecolor="none")
    x = np.linspace(data.quantile(0.001), data.quantile(0.999), 300)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()),
            "k--", linewidth=1.4, label="Normal")
    ax.set_title(f"{col}\nKurt={stats.kurtosis(data):.2f} Skew={stats.skew(data):.2f}",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

# Panel 4: Correlation heatmap
ax_c = fig.add_subplot(gs[3,0])
im = ax_c.imshow(corr, cmap="RdYlGn", vmin=-0.1, vmax=0.7, aspect="auto")
ax_c.set_xticks(range(3)); ax_c.set_yticks(range(3))
ax_c.set_xticklabels(list(COLORS.keys()), fontsize=9, rotation=15)
ax_c.set_yticklabels(list(COLORS.keys()), fontsize=9)
for i in range(3):
    for j in range(3):
        ax_c.text(j,i,f"{corr.iloc[i,j]:.3f}",ha="center",va="center",
                  fontsize=10,fontweight="bold",
                  color="white" if corr.iloc[i,j]>0.5 else "black")
ax_c.set_title("Correlation Matrix\n(2015–2024)", fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax_c, fraction=0.046, pad=0.04)

# Panel 5: Rolling 60-day correlations
ax_r = fig.add_subplot(gs[3,1:])
ax_r.plot(ret.index, ret["MASI"].rolling(60).corr(ret["EGX30"]),
          color="#533AB7", label="MASI–EGX30", linewidth=1.0)
ax_r.plot(ret.index, ret["MASI"].rolling(60).corr(ret["Tunindex"]),
          color="#D85A30", label="MASI–Tunindex", linewidth=1.0)
ax_r.plot(ret.index, ret["EGX30"].rolling(60).corr(ret["Tunindex"]),
          color="#1D9E75", label="EGX30–Tunindex", linewidth=1.0)
for s,e,c,_ in PERIOD_SPANS:
    ax_r.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.35)
ax_r.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax_r.set_title("Rolling 60-Day Correlations", fontsize=10, fontweight="bold")
ax_r.legend(fontsize=8); ax_r.spines[["top","right"]].set_visible(False)

plt.savefig("data_overview.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ Saved: data_overview.png")

---
## Step 5 — Principal Component Analysis (PCA)

**What it does:** Finds the hidden common forces driving co-movement between markets.  
**PC1** = regional market factor | **PC2** = Egypt vs Tunisia contrast | **PC3** = Morocco-specific

In [ ]:
sp_results = {}
for label,(s,e) in PERIODS.items():
    sub  = ret.loc[s:e].dropna()
    pca  = PCA(n_components=3)
    scr  = pca.fit_transform(StandardScaler().fit_transform(sub))
    sp_results[label] = {
        "pca":pca, "n":len(sub), "sub":sub,
        "scores": pd.DataFrame(scr, index=sub.index, columns=["PC1","PC2","PC3"]),
        "ev":  pca.explained_variance_ratio_,
        "eigs":pca.explained_variance_,
        "loadings": pd.DataFrame(pca.components_.T,
                                 index=["MASI","EGX30","Tunindex"],
                                 columns=["PC1","PC2","PC3"]),
    }
    lbl = label.replace("\n"," ")
    ev  = pca.explained_variance_ratio_
    print(f"  {lbl:25s} PC1={ev[0]*100:.1f}%  PC2={ev[1]*100:.1f}%  PC3={ev[2]*100:.1f}%")

full = sp_results["Full Sample\n(2015–2024)"]

# Save eigenvalue table
pd.DataFrame([{"Component":f"PC{i+1}",
               "Eigenvalue":round(full['eigs'][i],4),
               "Var.(%)":round(full['ev'][i]*100,2),
               "Cumul.(%)":round(sum(full['ev'][:i+1])*100,2)}
              for i in range(3)]).to_csv("pca_eigenvalues.csv", index=False)

print("\nFull-sample factor loadings:")
print(full["loadings"].round(4).to_string())

---
## Step 6 — PCA Figures (scree, stability, biplot, PC1 time series, AFC)

In [ ]:
# ── Scree plot ────────────────────────────────────────────
fig, axes = plt.subplots(1,2, figsize=(12,5), facecolor="white")
evs = full["ev"]*100
bars = axes[0].bar(["PC1","PC2","PC3"], evs, color=PC_COLORS, alpha=0.85, edgecolor="white", width=0.5)
axes[0].plot(["PC1","PC2","PC3"], evs, "o--", color="#333", linewidth=1.5)
for bar,v in zip(bars,evs):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
axes[0].axhline(33.3, color="gray", linestyle=":", label="Equal share (33.3%)")
axes[0].set_title("Scree Plot — Full Sample", fontsize=12, fontweight="bold"); axes[0].set_ylim(0,70)
axes[0].legend(fontsize=9); axes[0].spines[["top","right"]].set_visible(False)
cumev = np.cumsum(evs)
axes[1].bar(["PC1","PC2","PC3"], cumev, color=PC_COLORS, alpha=0.5, edgecolor="white", width=0.5)
axes[1].plot(["PC1","PC2","PC3"], cumev, "s-", color="#333", linewidth=2, markersize=8)
for i,v in enumerate(cumev):
    axes[1].text(i, v+1.5, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
axes[1].axhline(80, color="#E63946", linestyle="--", linewidth=1.2, label="80% threshold")
axes[1].set_title("Cumulative Variance Explained", fontsize=12, fontweight="bold"); axes[1].set_ylim(0,115)
axes[1].legend(fontsize=9); axes[1].spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("pca_scree.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ pca_scree.png")

# ── Stability bar chart ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9,5), facecolor="white")
labels_c = ["Full\nSample","Pre-COVID\n(2015–19)","COVID\n(2020–21)","Post-COVID\n(2022–24)"]
x = np.arange(len(labels_c)); w = 0.25
for i,(pc,color) in enumerate(zip(["PC1","PC2","PC3"],PC_COLORS)):
    vals = [sp_results[k]["ev"][i]*100 for k in PERIODS]
    bars = ax.bar(x+(i-1)*w, vals, w, label=pc, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{bar.get_height():.1f}%", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels_c, fontsize=10)
ax.set_title("Variance Explained per PC — Stability Across Sub-Periods", fontsize=12, fontweight="bold")
ax.set_ylabel("Variance Explained (%)"); ax.set_ylim(0,75)
ax.legend(fontsize=10); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("pca_stability.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ pca_stability.png")

# ── Rolling PCA ───────────────────────────────────────────
window=120; roll_pc1=[]; roll_pc2=[]; roll_pc3=[]; roll_dates=[]
ret_clean = ret.dropna()
for i in range(window, len(ret_clean)):
    sub = ret_clean.iloc[i-window:i]
    p   = PCA(n_components=3); p.fit(StandardScaler().fit_transform(sub))
    roll_pc1.append(p.explained_variance_ratio_[0]*100)
    roll_pc2.append(p.explained_variance_ratio_[1]*100)
    roll_pc3.append(p.explained_variance_ratio_[2]*100)
    roll_dates.append(ret_clean.index[i])
roll_df = pd.DataFrame({"PC1":roll_pc1,"PC2":roll_pc2,"PC3":roll_pc3}, index=roll_dates)
fig, ax = plt.subplots(figsize=(13,5), facecolor="white")
ax.fill_between(roll_df.index, roll_df["PC1"], alpha=0.2, color=PC_COLORS[0])
ax.plot(roll_df.index, roll_df["PC1"], color=PC_COLORS[0], linewidth=1.2, label="PC1 (common factor)")
ax.plot(roll_df.index, roll_df["PC2"], color=PC_COLORS[1], linewidth=0.9, alpha=0.8, label="PC2")
ax.plot(roll_df.index, roll_df["PC3"], color=PC_COLORS[2], linewidth=0.9, alpha=0.8, label="PC3")
for s,e,c,lbl in PERIOD_SPANS:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.4, label=lbl)
ax.axhline(roll_df["PC1"].mean(), color=PC_COLORS[0], linewidth=1, linestyle="--", alpha=0.7,
           label=f"PC1 mean ({roll_df['PC1'].mean():.1f}%)")
ax.set_title("Rolling 120-Day PCA — Variance Explained Over Time", fontsize=12, fontweight="bold")
ax.set_ylabel("Variance Explained (%)"); ax.set_ylim(20,75)
ax.legend(fontsize=8, ncol=4); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("rolling_pca.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ rolling_pca.png")

# ── AFC Pearson residuals ──────────────────────────────────
def discretise(series, low=-0.005, high=0.005):
    return pd.cut(series, bins=[-np.inf,low,high,np.inf], labels=["Negative","Neutral","Positive"])

disc = pd.DataFrame({col:discretise(ret[col].dropna()) for col in ret.columns}).dropna()
pairs = [("MASI","EGX30"),("MASI","Tunindex"),("EGX30","Tunindex")]
afc_results = {}
fig, axes = plt.subplots(1,3, figsize=(15,5), facecolor="white")
for ax,(a,b) in zip(axes,pairs):
    ct = pd.crosstab(disc[a], disc[b])
    chi2,p,dof,expected = stats.chi2_contingency(ct)
    residuals = (ct.values-expected)/np.sqrt(expected)
    afc_results[(a,b)] = {"ct":ct,"chi2":chi2,"p":p,"residuals":residuals}
    ct.to_csv(f"contingency_{a}_{b}.csv")
    im = ax.imshow(residuals, cmap="RdYlGn", vmin=-5, vmax=5, aspect="auto")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(["Neg","Neut","Pos"], fontsize=8, rotation=15)
    ax.set_yticklabels(["Neg","Neut","Pos"], fontsize=8)
    ax.set_xlabel(b,fontsize=10); ax.set_ylabel(a,fontsize=10)
    for r in range(3):
        for c in range(3):
            val=residuals[r,c]
            ax.text(c,r,f"{val:+.2f}",ha="center",va="center",fontsize=9,fontweight="bold",
                    color="white" if abs(val)>3 else "black")
    p_str="<0.001" if p<0.001 else f"={p:.3f}"
    ax.set_title(f"{a}×{b}\nχ²={chi2:.1f}, p{p_str}", fontsize=10, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    print(f"  {a}×{b}: χ²={chi2:.2f}  p{'<0.001' if p<0.001 else f'={p:.4f}'}  → {'Associated ✓' if p<0.05 else 'Independent ✗'}")
fig.suptitle("AFC Pearson Residuals — Co-movement of Market States", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout(); plt.savefig("afc_residuals.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ afc_residuals.png")

---
## Step 7 — Multiple Correspondence Analysis (ACM)

**What it adds over AFC:** analyses all three markets simultaneously in one geometric map.  
**Read the map:** categories that appear close together co-occur frequently.

In [ ]:
disc_with_period = disc.copy()
disc_with_period["Period"] = returns.loc[disc.index,"Period"]
PERIOD_DOT_COLORS = {
    "Pre-COVID (2015–2019)": "#AAAADD",
    "COVID (2020–2021)":     "#DD8866",
    "Post-COVID (2022–2024)":"#88BBAA",
}

# ── Full-sample ACM ───────────────────────────────────────
mca = prince.MCA(n_components=2, random_state=42)
mca.fit(disc)
row_coords  = mca.row_coordinates(disc)
col_coords  = mca.column_coordinates(disc)
eigenvalues = mca.eigenvalues_
total_inertia = sum(eigenvalues)
dim1_pct = eigenvalues[0]/total_inertia*100
dim2_pct = eigenvalues[1]/total_inertia*100

print(f"ACM — Full Sample")
print(f"  Total inertia : {total_inertia:.4f}  (theoretical max = M/p - 1 = 9/3 - 1 = 2)")
print(f"  Dimension 1   : {eigenvalues[0]:.4f} ({dim1_pct:.1f}%)  Kaiser threshold: >1/p=0.333 → {'✓' if eigenvalues[0]>1/3 else '✗'}")
print(f"  Dimension 2   : {eigenvalues[1]:.4f} ({dim2_pct:.1f}%)  Kaiser threshold: >1/p=0.333 → {'✓' if eigenvalues[1]>1/3 else '✗'}")
print("\nCategory coordinates:")
print(col_coords.round(4).to_string())
col_coords.round(4).to_csv("acm_coordinates.csv")

# ── ACM Map figure ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,8), facecolor="white")
period_col = returns.loc[disc.index,"Period"].map(PERIOD_DOT_COLORS).fillna("#CCCCCC")
ax.scatter(row_coords.iloc[:,0], row_coords.iloc[:,1],
           c=period_col, alpha=0.15, s=6, linewidths=0, zorder=1)
for idx_name in ["MASI","EGX30","Tunindex"]:
    pts=[]
    for cat in ["Negative","Neutral","Positive"]:
        key = f"{idx_name}__{cat}"
        if key in col_coords.index:
            x,y = col_coords.loc[key,0], col_coords.loc[key,1]
            ax.scatter(x,y, color=COLORS[idx_name], marker=INDEX_MARKERS[idx_name],
                       s=180, zorder=5, edgecolors="white", linewidth=1.2)
            off = 0.04 if x>=0 else -0.04
            ax.text(x+off, y+0.05, f"{idx_name}\n{cat}", fontsize=8, fontweight="bold",
                    color=COLORS[idx_name], ha="center", va="bottom",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")])
            pts.append((x,y))
    if len(pts)==3:
        ax.plot([p[0] for p in pts],[p[1] for p in pts],
                color=COLORS[idx_name], linewidth=1.2, linestyle="--", alpha=0.5, zorder=3)
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.set_xlabel(f"Dimension 1 ({dim1_pct:.1f}% of inertia)", fontsize=11)
ax.set_ylabel(f"Dimension 2 ({dim2_pct:.1f}% of inertia)", fontsize=11)
ax.set_title("ACM — Joint Market State Map\nMASI, EGX30 & Tunindex (Full Sample 2015–2024)",
             fontsize=12, fontweight="bold", pad=12)
idx_leg    = [Line2D([0],[0],marker=INDEX_MARKERS[k],color="w",markerfacecolor=COLORS[k],markersize=10,label=k)
              for k in ["MASI","EGX30","Tunindex"]]
period_leg = [Patch(facecolor=v, label=k) for k,v in PERIOD_DOT_COLORS.items()]
leg1 = ax.legend(handles=idx_leg, title="Index", fontsize=9, loc="upper right", framealpha=0.9)
ax.add_artist(leg1)
ax.legend(handles=period_leg, title="Period", fontsize=8, loc="lower right", framealpha=0.9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("acm_fullsample.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ acm_fullsample.png")

# ── ACM sub-period maps ───────────────────────────────────
period_labels = ["Pre-COVID (2015–2019)","COVID (2020–2021)","Post-COVID (2022–2024)"]
fig, axes = plt.subplots(1,3, figsize=(16,6), facecolor="white")
fig.suptitle("ACM — Market State Maps by Sub-Period", fontsize=13, fontweight="bold", y=1.02)
for ax,plabel in zip(axes,period_labels):
    sub_disc = disc_with_period[disc_with_period["Period"]==plabel][["MASI","EGX30","Tunindex"]]
    m = prince.MCA(n_components=2, random_state=42); m.fit(sub_disc)
    cc = m.column_coordinates(sub_disc); rc = m.row_coordinates(sub_disc)
    ev = m.eigenvalues_; tot = sum(ev)
    ax.scatter(rc.iloc[:,0], rc.iloc[:,1], color="#AAAAAA", alpha=0.12, s=6, linewidths=0)
    for idx_name in ["MASI","EGX30","Tunindex"]:
        pts=[]
        for cat in ["Negative","Neutral","Positive"]:
            key=f"{idx_name}__{cat}"
            if key in cc.index:
                x,y=cc.loc[key,0],cc.loc[key,1]
                ax.scatter(x,y, color=COLORS[idx_name], marker=INDEX_MARKERS[idx_name],
                           s=160, zorder=5, edgecolors="white", linewidth=1.2)
                off=0.04 if x>=0 else -0.04
                ax.text(x+off,y+0.05,f"{idx_name}\n{cat}",fontsize=7,fontweight="bold",
                        color=COLORS[idx_name],ha="center",va="bottom",
                        path_effects=[pe.withStroke(linewidth=2,foreground="white")])
                pts.append((x,y))
        if len(pts)==3:
            ax.plot([p[0] for p in pts],[p[1] for p in pts],
                    color=COLORS[idx_name],linewidth=1.2,linestyle="--",alpha=0.5,zorder=3)
    ax.axhline(0,color="gray",linewidth=0.4,linestyle="--",alpha=0.4)
    ax.axvline(0,color="gray",linewidth=0.4,linestyle="--",alpha=0.4)
    ax.set_xlabel(f"Dim 1 ({ev[0]/tot*100:.1f}%)",fontsize=9)
    ax.set_ylabel(f"Dim 2 ({ev[1]/tot*100:.1f}%)",fontsize=9)
    ax.set_title(f"{plabel}\n(n={len(sub_disc)})",fontsize=10,fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("acm_by_period.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show(); print("✓ acm_by_period.png")

---
## ✅ Summary — Files Generated

In [ ]:
import os
files = {
    "prices_raw.csv"               :"Raw closing prices",
    "log_returns_with_periods.csv" :"Daily log-returns + regime labels",
    "descriptive_stats.csv"        :"Table 1 — Descriptive statistics",
    "correlation_matrix.csv"       :"Table 2 — Correlation matrix",
    "pca_eigenvalues.csv"          :"Table 3 — PCA eigenvalues",
    "acm_coordinates.csv"          :"Table 5 — ACM category coordinates",
    "data_overview.png"            :"Figure 1 — EDA dashboard",
    "pca_scree.png"                :"Figure 2 — Scree plot",
    "pca_stability.png"            :"Figure 3 — PC stability chart",
    "rolling_pca.png"              :"Figure 4 — Rolling PCA",
    "afc_residuals.png"            :"Figure 5 — AFC residuals",
    "acm_fullsample.png"           :"Figure 6 — ACM full-sample map",
    "acm_by_period.png"            :"Figure 7 — ACM by sub-period",
}
print("="*60)
print("✓  ALL STEPS COMPLETE")
print("="*60)
for fname, desc in files.items():
    exists = os.path.exists(fname)
    print(f"  {'✓' if exists else '✗ MISSING':<10} {fname:<40} {desc}")